# Data Warehouse & Analytics Project Catalog
### Medallion Architecture.    

---

## 1. Project Overview

This project implements a modern SQL Server data warehouse using the Medallion architecture to consolidate CRM and ERP sales data into a trusted, analytics-ready model.

- Database: `DataWarehouse`
- Schemas: `bronze`, `silver`, `gold`
- Architecture: Bronze → Silver → Gold
- Modeling pattern: Star schema for analytics
- Primary business domain: Sales and customer/product performance analytics

### Business Goal
The warehouse is designed to:
- ingest raw source data from multiple systems,
- standardize and cleanse it in the silver layer,
- build curated analytical views in the gold layer,
- support BI reporting and business intelligence workflows.

---

## 2. Source Systems and Data Assets

| Source System | File/Asset | Schema | Description |
|---|---|---|---|
| CRM | `source_crm/cust_info.csv` | `bronze.crm_cust_info` | Customer master profile data |
| CRM | `source_crm/prd_info.csv` | `bronze.crm_prd_info` | Product master data |
| CRM | `source_crm/sales_details.csv` | `bronze.crm_sales_details` | Sales transactions and order events |
| ERP | `source_erp/CUST_AZ12.csv` | `bronze.erp_cust_az12` | ERP-based customer attributes |
| ERP | `source_erp/LOC_A101.csv` | `bronze.erp_loc_a101` | Country/location reference data |
| ERP | `source_erp/PX_CAT_G1V2.csv` | `bronze.erp_px_cat_g1v2` | Product category and subcategory reference |

### Source-to-Target Flow

```mermaid
flowchart LR
    A[CRM CSV Files] --> B[Bronze Layer]
    C[ERP CSV Files] --> B
    B --> D[Silver Layer]
    D --> E[Gold Layer]
    E --> F[BI / Reporting / Dashboards]
```

---

## 3. Data Architecture Summary

### Bronze Layer
Purpose: raw ingestion layer with minimal transformations.

### Silver Layer
Purpose: cleaned, standardized, business-ready data with consistent naming and quality handling.

### Gold Layer
Purpose: business-friendly analytical layer optimized for reporting and decision-making.

---

## 4. Bronze Layer Catalog

The bronze layer stores raw ingested tables exactly as captured from the source systems, before cleansing and transformation.

| Table | Description | Key Fields | Source |
|---|---|---|---|
| `bronze.crm_cust_info` | Raw CRM customer master data | `cst_id`, `cst_key`, `cst_firstname`, `cst_lastname` | CRM |
| `bronze.crm_prd_info` | Raw CRM product master data | `prd_id`, `prd_key`, `prd_nm`, `prd_cost` | CRM |
| `bronze.crm_sales_details` | Raw CRM sales transactions | `sls_ord_num`, `sls_prd_key`, `sls_cust_id`, `sls_sales` | CRM |
| `bronze.erp_cust_az12` | ERP customer detail supplement | `cid`, `bdate`, `gen` | ERP |
| `bronze.erp_loc_a101` | ERP location metadata | `cid`, `cntry` | ERP |
| `bronze.erp_px_cat_g1v2` | ERP category reference table | `id`, `cat`, `subcat`, `maintenance` | ERP |

### Bronze Column Notes
- All tables are designed as landing tables for source data ingestion.
- Data is stored in a raw, minimally processed format before validation.
- Some fields may contain nulls, inconsistencies, or naming differences that are resolved in silver.

---

## 5. Silver Layer Catalog

The silver layer transforms raw data into a standardized analytical foundation. It includes clean naming, data quality fixes, and load metadata such as `dwh_create_date`.

| Table | Description | Transformation Focus |
|---|---|---|
| `silver.crm_cust_info` | Standardized CRM customer dimension staging table | Rename consistency, load metadata, quality cleaning |
| `silver.crm_prd_info` | Standardized CRM product staging table | Active and historical product handling |
| `silver.crm_sales_details` | Standardized sales transaction staging table | Numeric and business key standardization |
| `silver.erp_cust_az12` | ERP customer enrichment table | Birthdate and gender enrichment |
| `silver.erp_loc_a101` | ERP geography table | Country dimension enrichment |
| `silver.erp_px_cat_g1v2` | ERP product category reference table | Category mapping and maintenance attributes |

### Silver Layer Data Quality Principles
- Schema alignment with downstream models
- Null handling and consistency checks
- Business-key normalization for joins
- Standardized timestamps and ingestion markers
- Support for downstream gold object creation

---

## 6. Gold Layer Catalog

The gold layer is the curated analytics layer used by BI tools and reporting teams. It contains business-friendly views designed for query performance and analytical readability.

### 6.1 Gold Dimension: `gold.dim_customers`

| Attribute | Type | Description |
|---|---|---|
| `customer_key` | Surrogate key | Generated via `ROW_NUMBER()` to support analytic joins |
| `customer_id` | Business key | CRM customer identifier |
| `customer_number` | Business key | Customer business / external number |
| `first_name` | Attribute | Customer first name |
| `last_name` | Attribute | Customer last name |
| `country` | Attribute | Country from ERP reference data |
| `marital_status` | Attribute | Customer marital status |
| `gender` | Attribute | Uses CRM value first, ERP fallback, default `n/a` |
| `birthdate` | Attribute | ERP birth date; default `1900-01-01` when null |
| `create_date` | Attribute | CRM creation date |

**Purpose:** one row per customer for segmentation, customer behavior, and demographic analysis.

**Source tables:**
- `silver.crm_cust_info`
- `silver.erp_cust_az12`
- `silver.erp_loc_a101`

**Business logic:**
- CRM is treated as master source for customer profile data.
- ERP data enriches gap areas like birthdate, gender, and country.

---

### 6.2 Gold Dimension: `gold.dim_products`

| Attribute | Type | Description |
|---|---|---|
| `product_key` | Surrogate key | Generated via `ROW_NUMBER()` |
| `product_id` | Business key | CRM product identifier |
| `product_number` | Business key | Product code / product number |
| `product_name` | Attribute | Product name |
| `catagory_id` | Attribute | Product category ID |
| `catagory` | Attribute | Category label |
| `subcatagory` | Attribute | Subcategory label |
| `maintenance` | Attribute | Product maintenance metadata |
| `cost` | Measure | Product cost |
| `product_line` | Attribute | Product line / family |
| `start_date` | Attribute | Product start date |

**Purpose:** one row per active product used for product performance and category analysis.

**Source tables:**
- `silver.crm_prd_info`
- `silver.erp_px_cat_g1v2`

**Business logic:**
- Only active products are included using `WHERE prd_end_dt IS NULL`.
- Product category attributes are enriched from ERP reference data.

---

### 6.3 Gold Fact: `gold.fact_sales`

| Attribute | Type | Description |
|---|---|---|
| `order_number` | Foreign key / order identifier | Sales order number |
| `product_key` | Foreign key | Links to `gold.dim_products` |
| `customer_key` | Foreign key | Links to `gold.dim_customers` |
| `order_date` | Date | Date of order |
| `shipping_date` | Date | Shipment date |
| `due_date` | Date | Due date for delivery |
| `quantity` | Measure | Units sold |
| `price` | Measure | Unit price |
| `sales_amount` | Measure | Total sales value |

**Purpose:** transaction-level sales fact for revenue, trend, and performance analysis.

**Source tables:**
- `silver.crm_sales_details`
- `gold.dim_products`
- `gold.dim_customers`

**Star schema relationship:**
- `gold.fact_sales.product_key` → `gold.dim_products.product_key`
- `gold.fact_sales.customer_key` → `gold.dim_customers.customer_key`

---

## 7. Star Schema Logical Model

```mermaid
erDiagram
    DIM_CUSTOMERS ||--o{ FACT_SALES : relates
    DIM_PRODUCTS ||--o{ FACT_SALES : relates

    DIM_CUSTOMERS {
        int customer_key PK
        int customer_id
        string customer_number
        string first_name
        string last_name
        string country
        string marital_status
        string gender
        date birthdate
        date create_date
    }

    DIM_PRODUCTS {
        int product_key PK
        int product_id
        string product_number
        string product_name
        string catagory
        string subcatagory
        string product_line
        decimal cost
        date start_date
    }

    FACT_SALES {
        string order_number
        int product_key FK
        int customer_key FK
        date order_date
        date shipping_date
        date due_date
        int quantity
        decimal price
        decimal sales_amount
    }
```

---

## 8. Data Lineage Summary

### Ingestion
- Raw files are loaded to `bronze` tables.

### Standardization
- Values are cleaned and normalized in `silver`.

### Curated Analytics
- Business-friendly dimensions and fact views are created in `gold`.

### Reporting Consumption
- Business analysts and BI users consume the gold layer for dashboards, KPI analysis, and executive reporting.

---

## 9. Data Governance and Quality Notes

| Area | Standard |
|---|---|
| Data ownership | CRM and ERP domain sources |
| Layer responsibility | Bronze = raw; Silver = quality; Gold = curated analytics |
| Key governance principle | Report from trusted, validated data |
| Observability | Track nulls, missing dimensions, and key mismatches |
| Refresh rule | Gold layer is rebuilt after silver validation |

### Key Quality Assumptions
- Customer and product dimensions are treated as master reference data.
- Missing ERP gender values default to `n/a`.
- Missing birthdates are normalized to `1900-01-01`.
- Only active product records are included in the product dimension.

---

## 10. Project Catalog Status

This project catalog covers the complete warehouse flow across the three major layers:

- Bronze layer inventory
- Silver layer standardization
- Gold layer analytics model
- Star schema structure and lineage

This catalog serves as a documentation reference for data engineering, data modeling, and analytics use cases.

---

## 11. Recommended Future Extensions

- Add column-level catalog for each table and view
- Add business glossary and metric definitions
- Add refresh schedule and operational ownership
- Add data quality rule checks and threshold documentation

> End of full project-wide data catalog.